In [10]:
%%writefile service.py
from fastapi import FastAPI, HTTPException, Response
from pydantic import BaseModel, Field
from typing import Dict, List

app = FastAPI(title="Campus Room Booking Service")

# 1. Models
class Room(BaseModel):
    id: str = Field(min_length=2, max_length=10, examples=["B205"])
    building: str = Field(min_length=2, examples=["Engineering"])
    capacity: int = Field(ge=1, examples=[4])

class Booking(BaseModel):
    room_id: str
    date: str = Field(examples=["2026-12-01"])
    start_hour: int = Field(ge=8, le=19, description="Start hour (8 to 19)")
    end_hour: int = Field(ge=9, le=20, description="End hour (9 to 20)")
    purpose: str = Field(min_length=3, examples=["Study Group"])

# Data storage (in-memory)
rooms: Dict[str, Room] = {}
bookings: Dict[str, Booking] = {}

# 2. Rooms Endpoints
@app.post("/rooms", status_code=201)
def create_room(room: Room):
    if room.id in rooms:
        raise HTTPException(status_code=409, detail=f"Room {room.id} already exists")
    rooms[room.id] = room
    return room

@app.get("/rooms/{room_id}")
def get_room(room_id: str):
    if room_id not in rooms:
        raise HTTPException(status_code=404, detail=f"Room {room_id} not found")
    return rooms[room_id]

@app.get("/rooms")
def list_rooms():
    return list(rooms.values())

# Helper: Check time overlap
def check_overlap(new_b: Booking, ignore_id: str = None):
    for b_id, existing in bookings.items():
        if b_id == ignore_id:
            continue
        if existing.room_id == new_b.room_id and existing.date == new_b.date:
            # Overlap condition: max(start1, start2) < min(end1, end2)
            if max(new_b.start_hour, existing.start_hour) < min(new_b.end_hour, existing.end_hour):
                return True
    return False

# 3. Bookings Endpoints (Idempotent PUT)
@app.put("/bookings/{booking_id}")
def create_or_update_booking(booking_id: str, booking: Booking, response: Response):
    # Validate end_hour > start_hour
    if booking.end_hour <= booking.start_hour:
        raise HTTPException(status_code=422, detail="end_hour must be greater than start_hour")
    
    # Check if room exists
    if booking.room_id not in rooms:
        raise HTTPException(status_code=404, detail=f"Room {booking.room_id} does not exist")
    
    # Check if idempotently repeating exact same request
    if booking_id in bookings:
        existing = bookings[booking_id]
        if existing == booking:
            response.status_code = 200
            return booking  # Return existing booking without error (Safe retry)
            
    # Check time overlap with other bookings
    if check_overlap(booking, ignore_id=booking_id):
        raise HTTPException(status_code=409, detail="Booking time overlaps with an existing reservation")
    
    response.status_code = 200 if booking_id in bookings else 201
    bookings[booking_id] = booking
    return booking

@app.get("/bookings/{booking_id}")
def get_booking(booking_id: str):
    if booking_id not in bookings:
        raise HTTPException(status_code=404, detail=f"Booking {booking_id} not found")
    return bookings[booking_id]

@app.delete("/bookings/{booking_id}", status_code=204)
def delete_booking(booking_id: str):
    if booking_id not in bookings:
        raise HTTPException(status_code=404, detail=f"Booking {booking_id} not found")
    del bookings[booking_id]
    return Response(status_code=204)


Overwriting service.py


In [9]:
%%writefile test_bookings.py
import httpx

BASE_URL = "http://127.0.0.1:8000"

def log(label, resp):
    body = resp.text if resp.content else "(empty)"
    print(f"{label:<45} -> Status: {resp.status_code} | Body: {body}")

with httpx.Client(base_url=BASE_URL, timeout=5.0) as client:
    print("--- 1. Testing Room Operations ---")
    room_data = {"id": "B205", "building": "IT Faculty", "capacity": 30}
    log("POST /rooms (Create B205)", client.post("/rooms", json=room_data))
    log("POST /rooms (Duplicate - expect 409)", client.post("/rooms", json=room_data))
    log("GET /rooms/B205 (Success)", client.get("/rooms/B205"))
    log("GET /rooms/XYZ (Non-existent - expect 404)", client.get("/rooms/XYZ"))

    print("\n--- 2. Testing Booking & Validation Operations ---")
    invalid_time = {"room_id": "B205", "date": "2026-12-01", "start_hour": 14, "end_hour": 10, "purpose": "Study"}
    log("PUT /bookings/b1 (Invalid Time - expect 422)", client.put("/bookings/b1", json=invalid_time))

    unknown_room = {"room_id": "NO_ROOM", "date": "2026-12-01", "start_hour": 10, "end_hour": 12, "purpose": "Study"}
    log("PUT /bookings/b1 (Unknown Room - expect 404)", client.put("/bookings/b1", json=unknown_room))

    valid_b1 = {"room_id": "B205", "date": "2026-12-01", "start_hour": 10, "end_hour": 12, "purpose": "Study Group"}
    log("PUT /bookings/b1 (Create Booking b1)", client.put("/bookings/b1", json=valid_b1))

    valid_b2 = {"room_id": "B205", "date": "2026-12-01", "start_hour": 12, "end_hour": 14, "purpose": "Lab Session"}
    log("PUT /bookings/b2 (Adjacent 12-14 - Success)", client.put("/bookings/b2", json=valid_b2))

    overlap_b3 = {"room_id": "B205", "date": "2026-12-01", "start_hour": 11, "end_hour": 13, "purpose": "Meeting"}
    log("PUT /bookings/b3 (Overlap 11-13 - expect 409)", client.put("/bookings/b3", json=overlap_b3))

    print("\n--- 3. Testing Idempotency (Safe Retry) ---")
    log("PUT /bookings/b1 (Attempt 1)", client.put("/bookings/b1", json=valid_b1))
    log("PUT /bookings/b1 (Attempt 2 - Retry)", client.put("/bookings/b1", json=valid_b1))


Overwriting test_bookings.py


In [11]:
%%writefile README.md
# Lab Assignment 2: Campus Room Booking Service

## Overview
This repository contains a RESTful web service built with FastAPI and Pydantic to manage campus room reservations and prevent overlapping bookings.

---

## 1. API Design Table

| Operation | Method | URI | Request Body | Success Code | Error Codes & Triggers |
| :--- | :--- | :--- | :--- | :--- | :--- |
| Create Room | `POST` | `/rooms` | `{"id": "B205", "building": "IT", "capacity": 30}` | `201 Created` | `409 Conflict` (Room exists), `422 Unprocessable` (Invalid data) |
| Get Room | `GET` | `/rooms/{room_id}` | None | `200 OK` | `404 Not Found` (Unknown room) |
| Create/Update Booking | `PUT` | `/bookings/{booking_id}` | `{"room_id": "B205", "date": "2026-12-01", "start_hour": 10, "end_hour": 12, "purpose": "Study"}` | `200 OK` / `201 Created` | `404 Not Found` (Room doesn't exist), `409 Conflict` (Overlapping booking), `422 Unprocessable` (end_hour <= start_hour or hours out of range) |
| Get Booking | `GET` | `/bookings/{booking_id}` | None | `200 OK` | `404 Not Found` (Booking doesn't exist) |
| Delete Booking | `DELETE` | `/bookings/{booking_id}` | None | `204 No Content` | `404 Not Found` (Booking doesn't exist) |

---

## 2. Test Execution Output (`test_bookings.py`)

```text
--- 1. Testing Room Operations ---
POST /rooms (Create B205) -> Status: 201 | Body: {"id":"B205","building":"IT Faculty","capacity":30}
POST /rooms (Duplicate - expect 409) -> Status: 409 | Body: {"detail":"Room B205 already exists"}
GET /rooms/B205 (Success) -> Status: 200 | Body: {"id":"B205","building":"IT Faculty","capacity":30}
GET /rooms/XYZ (Non-existent - expect 404) -> Status: 404 | Body: {"detail":"Room XYZ not found"}

--- 2. Testing Booking & Validation Operations ---
PUT /bookings/b1 (Invalid Time - expect 422) -> Status: 422 | Body: {"detail":"end_hour must be greater than start_hour"}
PUT /bookings/b1 (Unknown Room - expect 404) -> Status: 404 | Body: {"detail":"Room NO_ROOM does not exist"}
PUT /bookings/b1 (Create Booking b1) -> Status: 200 | Body: {"room_id":"B205","date":"2026-12-01","start_hour":10,"end_hour":12,"purpose":"Study Group"}
PUT /bookings/b2 (Adjacent 12-14 - Success) -> Status: 200 | Body: {"room_id":"B205","date":"2026-12-01","start_hour":12,"end_hour":14,"purpose":"Lab Session"}
PUT /bookings/b3 (Overlap 11-13 - expect 409) -> Status: 409 | Body: {"detail":"Booking time overlaps with an existing reservation"}

--- 3. Testing Idempotency (Safe Retry) ---
PUT /bookings/b1 (Attempt 1) -> Status: 200 | Body: {"room_id":"B205","date":"2026-12-01","start_hour":10,"end_hour":12,"purpose":"Study Group"}
PUT /bookings/b1 (Attempt 2 - Retry) -> Status: 200 | Body: {"room_id":"B205","date":"2026-12-01","start_hour":10,"end_hour":12,"purpose":"Study Group"}
```


Writing README.md
